# 19.3 Counterfactual explanations

A counterfactual asks for the smallest actionable change that would have changed the model decision.

The core audit is not just distance to a boundary; it is distance under mutable features, scaled costs, and plausibility checks.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_ladder_classifier(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr_s, y_tr)
    return clf, scaler, x_tr_s, x_te_s, y_tr, y_te


def score_for_class(model, X, class_index=None):
    proba = model.predict_proba(X)
    if class_index is None:
        class_index = int(np.argmax(proba[0]))
    return proba[:, class_index]


def finite_gradient(fn, x, eps=1e-4):
    grad = np.zeros_like(x, dtype=float)
    for j in range(len(x)):
        plus = x.copy()
        minus = x.copy()
        plus[j] = plus[j] + eps
        minus[j] = minus[j] - eps
        grad[j] = (fn(plus) - fn(minus)) / (2.0 * eps)
    return grad


def rank_corr(a, b):
    ar = pd.Series(a).rank(method="average").to_numpy()
    br = pd.Series(b).rank(method="average").to_numpy()
    if np.std(ar) == 0.0 or np.std(br) == 0.0:
        return 0.0
    return float(np.corrcoef(ar, br)[0, 1])


def top_k_mask(values, k):
    order = np.argsort(-np.abs(values))
    mask = np.zeros(len(values), dtype=bool)
    mask[order[:k]] = True
    return mask


## The concept, built once (D1)

A counterfactual solves $x_{cf}=rg\min_z d(z,x)\;	ext{s.t.}\; f(z)=y_{target}$. We search candidate edits, enforce mutability, and assert the lesson's cost receipt.

In [ ]:

def counterfactual_search(x, target, mutable_mask, cost, score_fn, candidates):
    best = None
    best_cost = np.inf
    for z in candidates:
        changed_immutable = np.any(np.abs((z - x)[~mutable_mask]) > 1e-9)
        if changed_immutable:
            continue
        pred = int(score_fn(z) >= 0.5)
        if pred != target:
            continue
        distance = float(np.sum(cost * np.abs(z - x)))
        if distance < best_cost:
            best = z.copy()
            best_cost = distance
    return best, best_cost


def toy_cf_score(z):
    return float(np.sum(z))

x_toy = np.zeros(3)
mutable_toy = np.array([True, True, True])
cost_toy = np.ones(3)
candidates_toy = np.array([
    [0.6, 0.3, 0.1],
    [1.2, 0.0, 0.0],
    [0.0, 1.2, 0.0],
])
x_cf_toy, cost_value_toy = counterfactual_search(x_toy, 1, mutable_toy, cost_toy, toy_cf_score, candidates_toy)
components_toy = cost_toy * np.abs(x_cf_toy - x_toy)
share_toy = float(np.max(components_toy) / components_toy.sum())
guarded_toy = cost_value_toy + 0.4 * components_toy.sum()
contrast_toy = cost_value_toy - components_toy[2]

assert np.allclose(x_cf_toy, [0.6, 0.3, 0.1])
assert np.isclose(cost_value_toy, 1.0)
assert np.isclose(components_toy.sum(), 1.0)
assert np.isclose(share_toy, 0.6)
assert np.isclose(guarded_toy, 1.4)
assert np.isclose(contrast_toy - cost_value_toy, -0.1)

print("counterfactual", x_cf_toy)
print("cost components", components_toy)
print("top share", share_toy)


For the ladder, candidate edits move along standardized feature axes toward the class-conditional target mean. The metric is the valid actionable counterfactual rate within a fixed distance budget.

In [ ]:

def classifier_counterfactual(model, x, target_class, mutable_mask, cost, target_mean, step_grid):
    candidates = []
    for amount in step_grid:
        z = x.copy()
        direction = target_mean - x
        z[mutable_mask] = x[mutable_mask] + amount * direction[mutable_mask]
        candidates.append(z)
    candidates = np.array(candidates)

    def score_fn(z):
        return float(score_for_class(model, z.reshape(1, -1), class_index=target_class)[0])

    best = None
    best_cost = np.inf
    for z in candidates:
        pred = int(np.argmax(model.predict_proba(z.reshape(1, -1))[0]))
        if pred != target_class:
            continue
        distance = float(np.sum(cost * np.abs(z - x)))
        if distance < best_cost:
            best = z.copy()
            best_cost = distance
    return best, best_cost


def valid_actionable_rate(model, x_te, y_te, x_tr, y_tr, budget=8.0):
    valid = 0
    total = min(20, len(x_te))
    distances = []
    for i in range(total):
        x = x_te[i]
        current = int(np.argmax(model.predict_proba(x.reshape(1, -1))[0]))
        classes = np.unique(y_tr)
        target = int(classes[classes != current][0])
        target_mean = np.mean(x_tr[y_tr == target], axis=0)
        mutable_mask = np.ones(x.shape[0], dtype=bool)
        if x.shape[0] > 2:
            mutable_mask[-1] = False
        cost = 1.0 / (np.std(x_tr, axis=0) + 0.25)
        best, distance = classifier_counterfactual(model, x, target, mutable_mask, cost, target_mean, np.linspace(0.1, 2.0, 30))
        if best is not None and distance <= budget:
            valid = valid + 1
            distances.append(distance)
    rate = valid / total
    mean_distance = float(np.mean(distances)) if distances else np.nan
    return rate, mean_distance


## The dataset ladder

We use the shared F15 classification ladder: a hand linear toy, clean blobs, a nonlinear moons stress test, real Wine, and real Breast Cancer. The same logistic base model and the same metric code run unchanged across rungs.

In [ ]:

rungs = clf_ladder()

for index, item in enumerate(rungs, start=1):
    name, X, y = item
    counts = dict(zip(*np.unique(y, return_counts=True)))
    print(index, name)
    print("shape", X.shape)
    print("class counts", counts)
    print("sample", np.round(X[:3, : min(5, X.shape[1])], 3))


## Run the SAME method across D1-D5

The metric is valid actionable counterfactual rate at a fixed cost budget.

In [ ]:

cf_rows = []
cf_artifacts = []

for rung, item in enumerate(rungs, start=1):
    name, X, y = item
    model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
    rate, mean_distance = valid_actionable_rate(model, x_te, y_te, x_tr, y_tr, budget=8.0)
    current = int(np.argmax(model.predict_proba(x_te[0].reshape(1, -1))[0]))
    target = int(np.unique(y_tr)[np.unique(y_tr) != current][0])
    target_mean = np.mean(x_tr[y_tr == target], axis=0)
    mutable_mask = np.ones(x_te.shape[1], dtype=bool)
    best, distance = classifier_counterfactual(model, x_te[0], target, mutable_mask, np.ones(x_te.shape[1]), target_mean, np.linspace(0.1, 2.0, 30))
    cf_rows.append({"rung": rung, "name": name, "valid_rate": rate, "mean_distance": mean_distance})
    cf_artifacts.append((name, x_te[0], best))

cf_table = pd.DataFrame(cf_rows)
print(cf_table.to_string(index=False))


## Results visualization

Panels show original-to-counterfactual edits for the first features. The curve reports valid-rate-vs-stress.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()

for ax, artifact in zip(axes[:5], cf_artifacts):
    name, original, best = artifact
    keep = min(10, len(original))
    if best is None:
        best = original.copy()
    ax.plot(np.arange(keep), original[:keep], marker="o", label="original")
    ax.plot(np.arange(keep), best[:keep], marker="x", label="counterfactual")
    ax.set_title(name[:26])
    ax.legend(fontsize=8)

axes[5].plot(cf_table["rung"], cf_table["valid_rate"], marker="o")
axes[5].set_title("valid rate vs stress")
axes[5].set_xlabel("rung")
axes[5].set_ylabel("valid actionable rate")
axes[5].set_ylim(0.0, 1.05)
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: minimizing raw distance without constraints can edit immutable or implausible features. The fix uses a mutable mask, scaled costs, and a plausibility check against training ranges.

In [ ]:

name, X, y = rungs[-1]
model, scaler, x_tr, x_te, y_tr, y_te = fit_ladder_classifier(X, y)
x = x_te[0]
current = int(np.argmax(model.predict_proba(x.reshape(1, -1))[0]))
target = int(np.unique(y_tr)[np.unique(y_tr) != current][0])
target_mean = np.mean(x_tr[y_tr == target], axis=0)
raw_mask = np.ones(x.shape[0], dtype=bool)
fixed_mask = raw_mask.copy()
fixed_mask[-5:] = False
raw_best, raw_distance = classifier_counterfactual(model, x, target, raw_mask, np.ones(x.shape[0]), target_mean, np.linspace(0.1, 2.0, 30))
scaled_cost = 1.0 / (np.std(x_tr, axis=0) + 0.25)
fixed_best, fixed_distance = classifier_counterfactual(model, x, target, fixed_mask, scaled_cost, target_mean, np.linspace(0.1, 2.0, 30))
low = np.percentile(x_tr, 1, axis=0)
high = np.percentile(x_tr, 99, axis=0)
plausible = fixed_best is not None and np.all(fixed_best >= low - 0.5) and np.all(fixed_best <= high + 0.5)

print("raw unconstrained distance", raw_distance)
print("fixed constrained scaled distance", fixed_distance)
print("fixed counterfactual plausible", plausible)


## Evaluate it + Practice

- Compare the reported valid actionable counterfactual rate with a no-skill baseline such as shuffled features, shuffled groups, or random rankings.
- Sanity check signs, denominators, and the background/reference point before trusting a pretty plot.
- Ablation: remove the mutable mask and scaled costs, then identify the features the shortcut edited
- Failure signals: unstable ranks under small perturbations, a metric that improves while accuracy collapses, or explanations that change when the seed changes.
- For gap topics, especially influence functions, keep the lesson numbers visible and treat the notebook as an audit scaffold until the lesson body grows more examples.

Practice 1: Change the trust knob or kernel width and predict which metric should move before running it.

Practice 2: Swap the D5 background/reference set and explain which conclusion is no longer stable.

Practice 3: Turn the pitfall back on, then write one sentence explaining why the fixed version is safer.